# Phase 3 — The core 2³ factorial + interaction analysis

Completes the cube: Phase 2 measured the four single-factor corners (baseline, W, K, S) at
repeat 0; this runbook adds the four interaction corners (**W+S, K+S, W+K+S, W+K**) and then
the repeats. **All cells on A100** (IMPLEMENTATION_PLAN Phase 3, GPU-confound correction).

Split into three independently resumable sessions, ordered so a dead session still leaves
maximally interpretable data:

| session | cells | est. units (real Phase-2 rate: 11.73 u / 48 cells ≈ 0.24 u/cell) |
|---|---|---|
| A — interaction corners @ r0 | 48 (4 launches) | ~12–15 → completes all 8 corners: first full interaction matrix (point estimates) |
| B — repeat 1, all 8 corners | 96 (8 launches) | ~25–30 → 2-repeat spread on every effect |
| C — repeat 2, all 8 corners | 96 (8 launches) | ~25–30 → full 3-repeat spread |

Session-A corner order is deliberate (see the marginals data, `phase2_results/`):
**ws first** (headline W×S cell — both factors erode to ~1.0x by conc 64 alone, and this
pairing is the one GPU-untested combination), **ks second** (the genuinely-uncertain
QuantSpec cell: tau under quantized KV — Phase 2 showed tau is flat in concurrency, so any
tau movement here is attributable to KV quant), **wks third** (completes the interference
gap), **wk last** (predicted near-additive control).

**Requirements:** A100 runtime (verify below), `HF_TOKEN` Colab secret with Llama-3.1
access. Expect vLLM's FP8-KV "accuracy drop without proper scaling factor" warning on
K-corners — known and documented (PREREQ_RESULTS Check 6).

### Hardware pinning (2026-07-09) — read before running

Colab's **High-RAM toggle pins the A100 variant**: OFF = 40GB, ON = 80GB (confirmed
empirically, PREREQ_RESULTS Check 1). **This notebook requires High-RAM ON (A100-80GB)**
— Phase 2's marginal corners were measured on the 80GB card, and Sessions A–C complete
the same cubes; mixing variants inside a cube confounds the effects with hardware exactly
like the rejected H100/A100 routing. `analysis/factorial.py` now emits a loud "MIXED
HARDWARE" warning if a cube ever mixes cards.

The K-stress addendum moved to its own notebook, `colab/phase3b_kstress_40gb.ipynb`,
which pins the **40GB** card (High-RAM OFF) where both KV ceilings fit in a small grid.

In [1]:
# 1. Verify the GPU actually assigned
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import subprocess
gpu = subprocess.run(["nvidia-smi","--query-gpu=name","--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
assert "A100" in gpu, "Not an A100 - reconnect until Colab honors the selection."
mem = int(subprocess.run(["nvidia-smi","--query-gpu=memory.total","--format=csv,noheader,nounits"],
                         capture_output=True, text=True).stdout.strip().splitlines()[0])
assert mem > 70000, ("Got the 40GB A100 (%d MiB). Toggle High-RAM ON and reconnect: "
                     "cube sessions must match the Phase-2 80GB card.") % mem
units_before = input("Compute-units balance right now: ")

NVIDIA A100-SXM4-80GB, 81920 MiB
Compute-units balance right now: 206.23


In [2]:
# 2. Repo + Spec-Bench (RAG passages; external/ is git-ignored)
%cd /content
!git clone https://github.com/manojarulmurugan/SpecDecoding-Study-vLLM-SGLang.git repo 2>/dev/null || (cd repo && git pull)
%cd /content/repo
!git clone --depth 1 https://github.com/hemingkx/Spec-Bench.git external/Spec-Bench 2>/dev/null || echo "Spec-Bench already present"
!test -f external/Spec-Bench/data/spec_bench/question.jsonl && echo "RAG data OK"

/content
/content/repo
RAG data OK


In [3]:
# 2b. Restore already-completed Phase-3 progress (if any), so the sweep
#     resumes instead of re-running cells you already paid for. Safe to run
#     even if phase3_results/ doesn't exist yet (fresh start).
import shutil, glob, os
if os.path.isdir("phase3_results"):
    shutil.copytree("phase3_results", "results", dirs_exist_ok=True)
    n = len(glob.glob("results/runs/*.json"))
    print(f"Restored {n} previously-completed run records into results/ -- "
          "the sweep will skip these and only run what remains.")
else:
    print("No phase3_results/ found in the repo -- starting fresh.")


Restored 173 previously-completed run records into results/ -- the sweep will skip these and only run what remains.


In [4]:
# 3. Isolated vLLM environment (PREREQ_RESULTS Check 6 recipe; ~6-8 min)
%pip install -q virtualenv
!virtualenv -q /content/vllm_env
!/content/vllm_env/bin/pip install -q vllm==0.24.0 datasets pyyaml requests pytest ninja
!/content/vllm_env/bin/python -c "import vllm; print('vllm', vllm.__version__)"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 92.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 470.6/470.6 kB 45.1 MB/s eta 0:00:00
vllm 0.24.0


In [6]:
# 4. HF auth
import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
print("token set:", bool(os.environ["HF_TOKEN"]))

token set: True


In [ ]:
# 4b. Pre-download every checkpoint ONCE, before any server launch.
# Two reasons: (1) tqdm is silent when stdout is redirected, so an
# in-sweep cold download produces a silent server log (the watchdog now
# tolerates this via HF-cache-growth detection, but pre-warming makes
# launches deterministic); (2) auth/rate-limit errors surface HERE with
# visible progress, not mid-sweep. Re-runs are no-ops on a warm cache.
!/content/vllm_env/bin/hf download meta-llama/Llama-3.1-8B-Instruct || /content/vllm_env/bin/huggingface-cli download meta-llama/Llama-3.1-8B-Instruct
!/content/vllm_env/bin/hf download hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4 || /content/vllm_env/bin/huggingface-cli download hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4
!/content/vllm_env/bin/hf download yuhuili/EAGLE3-LLaMA3.1-Instruct-8B || /content/vllm_env/bin/huggingface-cli download yuhuili/EAGLE3-LLaMA3.1-Instruct-8B

In [7]:
# 5. Harness self-check (~1 min, no GPU). Includes the verified-against-
#    synthetic-cubes factorial statistics (tests/test_factorial.py).
!PATH="/content/vllm_env/bin:$PATH" /content/vllm_env/bin/python -m pytest tests -q

........................................................................ [ 59%]
.................................................                        [100%]
121 passed in 12.16s


In [8]:
# 6. Sanity: session-A server commands (nothing launches)
!PATH="/content/vllm_env/bin:$PATH" /content/vllm_env/bin/python -m harness.sweep "configs/factorial/cube_ws_*_r0.yaml" "configs/factorial/cube_ks_*_r0.yaml" "configs/factorial/cube_wks_*_r0.yaml" "configs/factorial/cube_wk_*_r0.yaml" --results-dir results --dry-run 2>&1 | grep "server command"

[sweep] server command: vllm serve hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4 --host 127.0.0.1 --port 8000 --dtype float16 --gpu-memory-utilization 0.85 --max-model-len 8192 --seed 1234 --no-enable-prefix-caching --quantization awq_marlin --kv-cache-dtype fp8 --speculative-config {"draft_tensor_parallel_size": 1, "method": "eagle3", "model": "yuhuili/EAGLE3-LLaMA3.1-Instruct-8B", "num_speculative_tokens": 5}


In [9]:
# 7. SESSION A: interaction corners at repeat 0 (ws -> ks -> wks -> wk).
# Resumable; a disconnect costs one cell. ~12-15 units.
!PATH="/content/vllm_env/bin:$PATH" /content/vllm_env/bin/python -m harness.sweep "configs/factorial/cube_ws_*_r0.yaml" "configs/factorial/cube_ks_*_r0.yaml" "configs/factorial/cube_wks_*_r0.yaml" "configs/factorial/cube_wk_*_r0.yaml" --results-dir results

[sweep] 48 run(s) in 4 server group(s)
[sweep] skip (complete): core_factorial_meta-llama-3-1-8b-instruct-awq-int4_w4a16-fp16-eagle3_gsm8k_c1_r0
[sweep] skip (complete): core_factorial_meta-llama-3-1-8b-instruct-awq-int4_w4a16-fp16-eagle3_gsm8k_c32_r0
[sweep] skip (complete): core_factorial_meta-llama-3-1-8b-instruct-awq-int4_w4a16-fp16-eagle3_gsm8k_c64_r0
[sweep] skip (complete): core_factorial_meta-llama-3-1-8b-instruct-awq-int4_w4a16-fp16-eagle3_gsm8k_c8_r0
[sweep] skip (complete): core_factorial_meta-llama-3-1-8b-instruct-awq-int4_w4a16-fp16-eagle3_humaneval_c1_r0
[sweep] skip (complete): core_factorial_meta-llama-3-1-8b-instruct-awq-int4_w4a16-fp16-eagle3_humaneval_c32_r0
[sweep] skip (complete): core_factorial_meta-llama-3-1-8b-instruct-awq-int4_w4a16-fp16-eagle3_humaneval_c64_r0
[sweep] skip (complete): core_factorial_meta-llama-3-1-8b-instruct-awq-int4_w4a16-fp16-eagle3_humaneval_c8_r0
[sweep] skip (complete): core_factorial_meta-llama-3-1-8b-instruct-awq-int4_w4a16-fp16-eagle3

In [10]:
# 8. First full interaction matrix (point estimates). NOTE: the analysis
# needs the Phase-2 marginal records in the same results dir -- if this is a
# fresh session, restore them first (unzip phase2_results.zip into results/,
# or copy results/runs/*.json from the committed phase2_results/runs/).
!cp -n phase2_results/runs/*.json results/runs/ 2>/dev/null; echo "phase2 records merged"
!PATH="/content/vllm_env/bin:$PATH" /content/vllm_env/bin/python -m analysis.factorial results

phase2 records merged
# Core 2^3 factorial: log-space effects on goodput_tok_s

Effect columns: log-effect (multiplicative factor) [min..max across complete repeats]; '~0?' = spread straddles zero. Gap > 0 = sub-additive interference (see analysis/factorial.py docstring).

## gsm8k @ concurrency 1

| effect | estimate |
|---|---|
| W | +0.439 (x1.55) |
| K | -0.353 (x0.70) |
| S | +0.304 (x1.36) |
| WK | -0.091 (x0.91) |
| WS | -0.253 (x0.78) |
| KS | -0.258 (x0.77) |
| WKS | -0.059 (x0.94) |

**Interference gap**: naive x4.13 vs measured x1.39 -> gap +1.088 log (naive overestimates by x2.97)
Complete repeats: [0]

## gsm8k @ concurrency 8

| effect | estimate |
|---|---|
| W | +0.268 (x1.31) |
| K | -0.234 (x0.79) |
| S | +0.158 (x1.17) |
| WK | -0.019 (x0.98) |
| WS | -0.242 (x0.78) |
| KS | -0.163 (x0.85) |
| WKS | -0.001 (x1.00) |

**Interference gap**: naive x2.82 vs measured x1.21 -> gap +0.847 log (naive overestimates by x2.33)
Complete repeats: [0]

## gsm8k @ concurrency 32

|

In [11]:
# 9. SESSION B: repeat 1 for ALL 8 corners (96 cells, 8 launches, ~25-30
# units). Fine to run in a separate Colab session -- re-run cells 1-5 first.
!PATH="/content/vllm_env/bin:$PATH" /content/vllm_env/bin/python -m harness.sweep "configs/factorial/cube_*_r1.yaml" --results-dir results

[sweep] 96 run(s) in 8 server group(s)
[sweep] skip (complete): core_factorial_llama-3-1-8b-instruct_fp16-fp16-none_gsm8k_c1_r1
[sweep] skip (complete): core_factorial_llama-3-1-8b-instruct_fp16-fp16-none_gsm8k_c32_r1
[sweep] skip (complete): core_factorial_llama-3-1-8b-instruct_fp16-fp16-none_gsm8k_c64_r1
[sweep] skip (complete): core_factorial_llama-3-1-8b-instruct_fp16-fp16-none_gsm8k_c8_r1
[sweep] skip (complete): core_factorial_llama-3-1-8b-instruct_fp16-fp16-none_humaneval_c1_r1
[sweep] skip (complete): core_factorial_llama-3-1-8b-instruct_fp16-fp16-none_humaneval_c32_r1
[sweep] skip (complete): core_factorial_llama-3-1-8b-instruct_fp16-fp16-none_humaneval_c64_r1
[sweep] skip (complete): core_factorial_llama-3-1-8b-instruct_fp16-fp16-none_humaneval_c8_r1
[sweep] skip (complete): core_factorial_llama-3-1-8b-instruct_fp16-fp16-none_rag_shared_prefix_c1_r1
[sweep] skip (complete): core_factorial_llama-3-1-8b-instruct_fp16-fp16-none_rag_shared_prefix_c32_r1
[sweep] skip (complete): c

In [12]:
# 10. SESSION C: repeat 2 for ALL 8 corners (96 cells, ~25-30 units).
# Kill lever if units run short: skip C at conc 1 and 8 but KEEP conc 32 and
# 64 -- the crossovers live between 32 and 64 (W hits 1.01x, S hits 0.97x on
# GSM8K at c64), so that's where spread decides whether a sign claim holds.
# Selective globs for that: "configs/factorial/cube_*_c32_r2.yaml" "configs/factorial/cube_*_c64_r2.yaml"
!PATH="/content/vllm_env/bin:$PATH" /content/vllm_env/bin/python -m harness.sweep "configs/factorial/cube_*_r2.yaml" --results-dir results

[sweep] 96 run(s) in 8 server group(s)
[sweep] group 1/8: 12 pending run(s)
[sweep] server command: vllm serve meta-llama/Llama-3.1-8B-Instruct --host 127.0.0.1 --port 8000 --dtype float16 --gpu-memory-utilization 0.85 --max-model-len 8192 --seed 1234 --no-enable-prefix-caching
[sweep] waiting for server (log: results/server_logs/server_20260710_034957.log)
[run] core_factorial_llama-3-1-8b-instruct_fp16-fp16-none_gsm8k_c1_r2: 64 prompts, concurrency=1, max_new_tokens=256
  [load] 25/64 requests done
  [load] 50/64 requests done
[run] core_factorial_llama-3-1-8b-instruct_fp16-fp16-none_gsm8k_c1_r2 -> results/runs/core_factorial_llama-3-1-8b-instruct_fp16-fp16-none_gsm8k_c1_r2.json (status=ok, 90.6 tok/s mean/request, tau=None, acc=0.828)
[run] core_factorial_llama-3-1-8b-instruct_fp16-fp16-none_gsm8k_c32_r2: 320 prompts, concurrency=32, max_new_tokens=256
  [load] 25/320 requests done
  [load] 50/320 requests done
  [load] 75/320 requests done
  [load] 100/320 requests done
  [load] 12

In [13]:
# 11. Final analysis: effects with [min..max] spread across repeats,
# interference gap per workload x concurrency, machine-readable JSON.
!PATH="/content/vllm_env/bin:$PATH" /content/vllm_env/bin/python -m analysis.factorial results
!PATH="/content/vllm_env/bin:$PATH" /content/vllm_env/bin/python -m analysis.marginals results

# Core 2^3 factorial: log-space effects on goodput_tok_s

Effect columns: log-effect (multiplicative factor) [min..max across complete repeats]; '~0?' = spread straddles zero. Gap > 0 = sub-additive interference (see analysis/factorial.py docstring).

## gsm8k @ concurrency 1

| effect | estimate |
|---|---|
| W | +0.440 (x1.55) [+0.437..+0.441] |
| K | -0.354 (x0.70) [-0.361..-0.350] |
| S | +0.310 (x1.36) [+0.296..+0.319] |
| WK | -0.093 (x0.91) [-0.094..-0.092] |
| WS | -0.253 (x0.78) [-0.257..-0.246] |
| KS | -0.258 (x0.77) [-0.266..-0.253] |
| WKS | -0.061 (x0.94) [-0.062..-0.059] |

**Interference gap**: naive x4.15 vs measured x1.40 -> gap +1.087 log (naive overestimates by x2.97) [+1.069..+1.106 across repeats]
Complete repeats: [0, 1, 2]

## gsm8k @ concurrency 8

| effect | estimate |
|---|---|
| W | +0.265 (x1.30) [+0.259..+0.275] |
| K | -0.234 (x0.79) [-0.238..-0.229] |
| S | +0.156 (x1.17) [+0.153..+0.161] |
| WK | -0.020 (x0.98) [-0.025..-0.014] |
| WS | -0.245 (x0.78) [

In [14]:
# 12. Preserve everything
units_after = input("Compute-units balance now: ")
print("Units burned:", units_before, "->", units_after)
!zip -qr phase3_results.zip results
from google.colab import files
files.download("phase3_results.zip")

Compute-units balance now: 181.6
Units burned: 206.23 -> 181.6


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Reading the result

The factorial report is the project's headline artifact. Per workload × concurrency:

- **Interference gap**: naive speedup (product of the three marginal wins) vs measured
  full-stack speedup, and the overestimate factor. Phase-2 marginals predict the naive
  product at conc=1 is ~4.2–6.4x depending on workload — the gap line says how much of
  that survives stacking, and the trend across concurrency says whether interference
  grows with load (the thesis).
- **W×S contrast** (headline): negative = W and S cannibalize each other (Block 0's
  batch-1 finding, now with a concurrency axis). Compare S-on-W relative speedup vs
  Phase 2's S-on-FP16 column at each concurrency.
- **K×S contrast + tau in the ks/wks cells**: Phase 2 measured tau flat in concurrency
  (2.85/4.08/2.52 by workload). If tau moves in the K-on cells, that's the QuantSpec
  acceptance channel showing up (or failing to) under continuous batching — either way
  a reportable finding.
- **W×K contrast**: the pre-registered "boring control" — expect ~0 ('~0?' flag). If it
  isn't, inspect before celebrating: both marginals were near-null at c64, so a large
  W×K would smell like a harness artifact.
- **'~0?' flags**: any effect whose across-repeat spread straddles zero is not a claim.
  After session B you have 2-repeat spreads; after C, 3.

Afterwards: commit results + reports, update PREREQ_RESULTS Check 1 with this session's
burn, and what remains is Phase 4 (SGLang seam, optional) and Phase 5 (decision guide +
write-ups).